# yeast1: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('41_yeast1_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,57.943,46.904,66.132,36.361,92.417,64.389,60.486,78.399,52.424
smote,71.168,59.458,53.421,68.523,74.597,71.560,59.576,79.374,65.313
bsmote,69.511,57.303,51.182,66.899,73.507,70.203,59.199,79.090,63.407
ros,71.814,60.111,54.215,68.406,75.829,72.118,59.877,79.647,65.962
adasyn,71.512,59.709,51.735,71.777,71.943,71.860,59.856,79.474,65.610
casb,71.505,59.825,54.564,67.008,76.635,71.822,59.900,79.534,65.665
casb_relaxed,71.126,59.396,54.670,65.848,77.393,71.621,59.870,79.480,65.261
casb_edited,70.798,59.414,53.549,69.235,73.697,71.466,60.835,79.912,65.106
casb_ros,71.699,59.979,54.257,67.933,76.066,72.000,59.664,79.448,65.839


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,62.531,51.315,61.277,44.295,88.626,66.461,58.416,78.325,56.923
smote,70.140,57.714,50.227,68.068,72.417,70.243,53.219,76.849,63.927
bsmote,68.725,56.209,50.531,63.993,74.218,69.106,52.231,76.557,62.467
ros,69.083,56.488,49.719,65.628,72.891,69.259,54.108,76.607,62.785
adasyn,69.258,56.636,47.544,70.290,68.483,69.387,52.032,76.064,62.947
casb,70.262,58.123,52.850,64.803,76.351,70.577,55.122,77.581,64.192
casb_relaxed,69.334,57.110,52.644,63.390,76.730,70.060,55.990,77.691,63.222
casb_edited,70.841,58.589,48.882,73.525,68.531,71.028,55.162,77.931,64.715
casb_ros,69.014,56.560,51.312,63.297,75.592,69.445,55.138,77.272,62.787


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,65.167,55.218,67.501,46.967,90.711,68.839,64.048,81.021,60.193
smote,71.524,61.051,61.900,60.833,84.550,72.691,63.329,80.904,66.288
bsmote,71.497,60.691,60.141,61.643,83.175,72.409,63.578,80.995,66.094
ros,71.192,60.905,62.959,59.321,85.735,72.528,63.970,81.255,66.049
adasyn,72.028,61.141,59.608,63.041,82.512,72.776,63.489,80.752,66.584
casb,72.087,61.666,61.951,61.650,84.455,73.052,63.611,81.081,66.877
casb_relaxed,71.235,60.754,61.988,60.249,84.692,72.470,63.909,81.182,65.994
casb_edited,72.804,61.058,53.670,71.885,74.408,73.146,63.868,81.219,66.931
casb_ros,71.162,60.957,63.378,59.204,85.924,72.564,63.528,80.861,66.059


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,55.742,44.388,66.267,33.798,92.796,63.297,59.763,78.990,50.065
smote,70.075,58.144,53.281,64.806,76.114,70.460,59.897,79.009,64.109
bsmote,70.060,57.843,50.549,68.421,72.180,70.301,59.058,78.353,63.951
ros,69.971,57.866,52.654,65.038,75.735,70.386,59.786,78.981,63.918
adasyn,70.101,57.894,49.279,71.093,69.573,70.333,59.458,78.820,63.998
casb,69.174,57.193,54.118,62.012,77.962,69.987,59.990,79.133,63.183
casb_relaxed,69.569,57.494,52.779,64.575,75.592,70.083,59.817,79.030,63.532
casb_edited,69.891,57.645,47.285,74.476,66.019,70.248,60.654,79.306,63.768
casb_ros,69.152,57.164,54.071,62.011,77.915,69.963,60.046,79.154,63.158


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,60.346,49.456,65.294,40.355,91.137,65.746,60.678,79.184,54.901
smote,70.727,59.092,54.707,65.557,76.919,71.238,59.005,79.034,64.909
bsmote,69.948,58.012,53.101,65.239,75.770,70.505,58.516,78.749,63.980
ros,70.515,58.842,54.887,64.598,77.547,71.073,59.435,79.123,64.679
adasyn,70.725,58.845,52.042,69.050,73.128,71.089,58.709,78.777,64.785
casb,70.757,59.202,55.871,63.868,78.851,71.360,59.656,79.332,64.979
casb_relaxed,70.316,58.689,55.520,63.515,78.602,71.059,59.896,79.346,64.502
casb_edited,71.083,59.176,50.847,72.280,70.664,71.472,60.130,79.592,65.130
casb_ros,70.257,58.665,55.755,63.111,78.874,70.993,59.594,79.184,64.461


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,71.699,ros,71.814,-0.114,LOSS
1,f1,casb_ros,59.979,ros,60.111,-0.132,LOSS
2,precision,casb_relaxed,54.670,ros,54.215,0.455,WIN
3,recall,casb_edited,69.235,adasyn,71.777,-2.542,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,70.841,smote,70.140,0.701,WIN
1,f1,casb_edited,58.589,smote,57.714,0.875,WIN
2,precision,casb,52.850,bsmote,50.531,2.319,WIN
3,recall,casb_edited,73.525,adasyn,70.290,3.235,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,72.804,adasyn,72.028,0.775,WIN
1,f1,casb,61.666,adasyn,61.141,0.526,WIN
2,precision,casb_ros,63.378,ros,62.959,0.420,WIN
3,recall,casb_edited,71.885,adasyn,63.041,8.844,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,71.083,smote,70.727,0.357,WIN
1,f1,casb,59.202,smote,59.092,0.110,WIN
2,precision,casb,55.871,ros,54.887,0.984,WIN
3,recall,casb_edited,72.280,adasyn,69.050,3.230,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.610626,0.515152,0.739130,0.395349,0.943128,0.669238,0.634057,0.826739,0.562889,None
1,42,1,svm,smote,0.75,0.743983,0.654545,0.683544,0.627907,0.881517,0.754712,0.660431,0.844318,0.699264,None
2,42,1,svm,bsmote,0.75,0.737137,0.627027,0.585859,0.674419,0.805687,0.740053,0.658561,0.840130,0.682082,None
3,42,1,svm,ros,0.75,0.737062,0.646341,0.679487,0.616279,0.881517,0.748898,0.660475,0.839855,0.691702,None
4,42,1,svm,adasyn,0.75,0.745648,0.637838,0.595960,0.686047,0.810427,0.748237,0.680039,0.851868,0.691743,None
5,42,1,svm,casb,1.00,0.762601,0.659459,0.616162,0.709302,0.819905,0.764604,0.667855,0.840957,0.711030,None
6,42,1,svm,casb_relaxed,1.00,0.749260,0.638743,0.580952,0.709302,0.791469,0.750386,0.673306,0.844649,0.694002,None
7,42,1,svm,casb_edited,0.50,0.745980,0.658537,0.692308,0.627907,0.886256,0.757081,0.644224,0.839138,0.702258,None
8,42,1,svm,casb_ros,1.00,0.762601,0.659459,0.616162,0.709302,0.819905,0.764604,0.667635,0.841232,0.711030,None
9,42,1,knn,original,0.50,0.639026,0.534247,0.650000,0.453488,0.900474,0.676981,0.619365,0.794941,0.586636,None
